# Step 3 — RLHF (DPO + PPO-RLHF)
This notebook trains DPO and PPO-RLHF from preference datasets produced in Step 2.

Notes:
- Change `ENV_KEY` to switch environments (e.g., `cartpole`, `pendulum`).
- Dataset sizes K are inferred from pickle filenames (e.g., `_K50.pkl`).
- Multiple seeds are inferred from filenames (e.g., `seed3`).
- This is a code-only notebook; run cells manually.

In [ ]:
from __future__ import annotations

import os
import re
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

In [ ]:
# === Config ===
ENV_KEY = "cartpole"  # change to: cartpole | pendulum
ENV_ID_MAP = {"cartpole": "CartPole-v1", "pendulum": "Pendulum-v1"}
ENV_ID = ENV_ID_MAP.get(ENV_KEY, ENV_KEY)

DATASET_ROOT = Path("outputs/datasets")
OUTPUT_DIR = Path("outputs/part3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# DPO config
DPO_BETA = 0.1
DPO_EPOCHS = 5
DPO_BATCH_SIZE = 16
DPO_LR = 3e-4

# Reward model config
RM_EPOCHS = 10
RM_BATCH_SIZE = 16
RM_LR = 1e-3

# PPO-RLHF config
PPO_TIMESTEPS = 50_000
PPO_LR = 3e-4
PPO_N_STEPS = 1024
PPO_BATCH_SIZE = 64
PPO_GAMMA = 0.99

# Evaluation
EVAL_EPISODES = 20

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## Dataset discovery + normalization

In [ ]:
K_REGEX = re.compile(r"_K(\d+)", re.IGNORECASE)
SEED_REGEX = re.compile(r"seed(\d+)", re.IGNORECASE)


@dataclass
class PrefPair:
    s_pref: np.ndarray
    a_pref: np.ndarray
    s_rej: np.ndarray
    a_rej: np.ndarray


@dataclass
class PreferenceDataset:
    env_key: str
    k: int
    seed: int
    path: Path
    pairs: List[PrefPair]


def load_preference_datasets(env_key: str) -> List[PreferenceDataset]:
    pattern = f"{env_key.lower()}_seed*_K*.pkl"
    paths = sorted(DATASET_ROOT.glob(pattern))
    datasets: List[PreferenceDataset] = []
    for path in paths:
        m_k = K_REGEX.search(path.name)
        if not m_k:
            continue
        m_seed = SEED_REGEX.search(path.name)
        if not m_seed:
            raise ValueError(f"Missing seed in dataset filename: {path.name}")
        k = int(m_k.group(1))
        seed = int(m_seed.group(1))
        with open(path, "rb") as f:
            raw_pairs = pickle.load(f)
        pairs: List[PrefPair] = []
        for pair in raw_pairs:
            tau1 = pair["tau_1"]
            tau2 = pair["tau_2"]
            states1 = np.asarray([step[0] for step in tau1])
            actions1 = np.asarray([step[1] for step in tau1])
            states2 = np.asarray([step[0] for step in tau2])
            actions2 = np.asarray([step[1] for step in tau2])
            label = int(pair["label"])
            if label == 0:
                s_pref, a_pref = states1, actions1
                s_rej, a_rej = states2, actions2
            else:
                s_pref, a_pref = states2, actions2
                s_rej, a_rej = states1, actions1
            pairs.append(PrefPair(s_pref, a_pref, s_rej, a_rej))
        datasets.append(PreferenceDataset(env_key=env_key, k=k, seed=seed, path=path, pairs=pairs))
    return datasets


dataset_list = load_preference_datasets(ENV_KEY)
dataset_list

## DPO (offline policy optimization from preferences)

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim: int, action_dim: int, continuous: bool):
        super().__init__()
        self.continuous = continuous
        hidden = 64 # To match MlpPolicy's default model architecture for the actor network
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
        )
        if continuous:
            self.mean = nn.Linear(hidden, action_dim)
            self.log_std = nn.Parameter(torch.zeros(action_dim))
        else:
            self.logits = nn.Linear(hidden, action_dim)

    def forward(self, obs: torch.Tensor):
        h = self.net(obs)
        if self.continuous:
            mean = self.mean(h)
            log_std = self.log_std.expand_as(mean)
            return mean, log_std
        return self.logits(h)


def trajectory_logprob(policy: PolicyNet, states: np.ndarray, actions: np.ndarray) -> torch.Tensor:
    states_t = torch.tensor(states, dtype=torch.float32, device=DEVICE)
    if policy.continuous:
        actions_t = torch.tensor(actions, dtype=torch.float32, device=DEVICE)
        mean, log_std = policy(states_t)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mean, std)
        logp = dist.log_prob(actions_t).sum(dim=-1)
    else:
        actions_t = torch.tensor(actions, dtype=torch.long, device=DEVICE)
        logits = policy(states_t)
        dist = torch.distributions.Categorical(logits=logits)
        logp = dist.log_prob(actions_t)
    return logp.sum()


def dpo_loss(
    policy: PolicyNet,
    ref_policy: PolicyNet,
    s_pref: np.ndarray,
    a_pref: np.ndarray,
    s_rej: np.ndarray,
    a_rej: np.ndarray,
    beta: float,
) -> torch.Tensor:
    logp_pref = trajectory_logprob(policy, s_pref, a_pref)
    logp_rej = trajectory_logprob(policy, s_rej, a_rej)
    logp_ref_pref = trajectory_logprob(ref_policy, s_pref, a_pref)
    logp_ref_rej = trajectory_logprob(ref_policy, s_rej, a_rej)
    adv = (logp_pref - logp_rej) - (logp_ref_pref - logp_ref_rej)
    loss = -F.logsigmoid(beta * adv)
    return loss.mean()


def train_dpo(
    pairs: List[PrefPair],
    obs_dim: int,
    action_dim: int,
    continuous: bool,
    epochs: int = DPO_EPOCHS,
    batch_size: int = DPO_BATCH_SIZE,
    lr: float = DPO_LR,
    beta: float = DPO_BETA,
) -> PolicyNet:
    policy = PolicyNet(obs_dim, action_dim, continuous).to(DEVICE)
    ref_policy = PolicyNet(obs_dim, action_dim, continuous).to(DEVICE)
    ref_policy.load_state_dict(policy.state_dict())
    ref_policy.eval()
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    n = len(pairs)
    for _ in range(epochs):
        idx = np.random.permutation(n)
        for start in range(0, n, batch_size):
            batch_idx = idx[start:start + batch_size]
            losses = []
            for i in batch_idx:
                p = pairs[i]
                loss = dpo_loss(
                    policy,
                    ref_policy,
                    p.s_pref,
                    p.a_pref,
                    p.s_rej,
                    p.a_rej,
                    beta,
                )
                losses.append(loss)
            if not losses:
                continue
            batch_loss = torch.stack(losses).mean()
            optimizer.zero_grad()
            batch_loss.backward()
            optimizer.step()
    return policy

## PPO-RLHF (reward model + PPO)

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, obs_dim: int, action_dim: int, continuous: bool):
        super().__init__()
        self.continuous = continuous
        hidden = 64
        in_dim = obs_dim + action_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )
        self.action_dim = action_dim

    def forward(self, obs: torch.Tensor, act: torch.Tensor) -> torch.Tensor:
        if self.continuous:
            act_feat = act
        else:
            act_feat = F.one_hot(act, num_classes=self.action_dim).float()
        x = torch.cat([obs, act_feat], dim=-1)
        return self.net(x).squeeze(-1)

    def trajectory_return(self, states: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
        rewards = self.forward(states, actions)
        return rewards.sum()


def reward_model_loss(
    model: RewardModel,
    s1_list: List[np.ndarray],
    a1_list: List[np.ndarray],
    s2_list: List[np.ndarray],
    a2_list: List[np.ndarray],
    labels: List[int],
) -> Tuple[torch.Tensor, torch.Tensor]:
    logits = []
    for s1, a1, s2, a2 in zip(s1_list, a1_list, s2_list, a2_list):
        s1_t = torch.tensor(s1, dtype=torch.float32, device=DEVICE)
        a1_t = torch.tensor(a1, dtype=torch.float32 if model.continuous else torch.long, device=DEVICE)
        s2_t = torch.tensor(s2, dtype=torch.float32, device=DEVICE)
        a2_t = torch.tensor(a2, dtype=torch.float32 if model.continuous else torch.long, device=DEVICE)
        r1 = model.trajectory_return(s1_t, a1_t)
        r2 = model.trajectory_return(s2_t, a2_t)
        logits.append(r1 - r2)
    logits_t = torch.stack(logits)
    labels_t = torch.tensor(labels, dtype=torch.float32, device=DEVICE)
    loss = F.binary_cross_entropy_with_logits(logits_t, labels_t)
    preds = (torch.sigmoid(logits_t) >= 0.5).float()
    acc = (preds == labels_t).float().mean()
    return loss, acc


def _pairs_to_lists(pairs: List[PrefPair]) -> Tuple[List[np.ndarray], List[np.ndarray], List[np.ndarray], List[np.ndarray], List[int]]:
    s1_list, a1_list, s2_list, a2_list = [], [], [], []
    for p in pairs:
        s1_list.append(p.s_pref)
        a1_list.append(p.a_pref)
        s2_list.append(p.s_rej)
        a2_list.append(p.a_rej)
    labels = [1] * len(pairs)
    return s1_list, a1_list, s2_list, a2_list, labels


def eval_reward_model(model: RewardModel, pairs: List[PrefPair]) -> Tuple[float, float]:
    if not pairs:
        return float("nan"), float("nan")
    s1_list, a1_list, s2_list, a2_list, labels = _pairs_to_lists(pairs)
    with torch.no_grad():
        loss, acc = reward_model_loss(model, s1_list, a1_list, s2_list, a2_list, labels)
    return float(loss.item()), float(acc.item())


def train_reward_model(
    pairs: List[PrefPair],
    obs_dim: int,
    action_dim: int,
    continuous: bool,
    epochs: int = RM_EPOCHS,
    batch_size: int = RM_BATCH_SIZE,
    lr: float = RM_LR,
    val_frac: float = 0.2,
    seed: int = 0,
    report_every: int = 1,
) -> RewardModel:
    model = RewardModel(obs_dim, action_dim, continuous).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(pairs)
    rng = np.random.default_rng(seed)
    order = rng.permutation(n)
    split = int(n * (1.0 - val_frac))
    train_pairs = [pairs[i] for i in order[:split]]
    val_pairs = [pairs[i] for i in order[split:]]
    for epoch in range(epochs):
        idx = np.random.permutation(len(train_pairs))
        for start in range(0, len(train_pairs), batch_size):
            batch_idx = idx[start:start + batch_size]
            s1_list, a1_list, s2_list, a2_list, labels = [], [], [], [], []
            for i in batch_idx:
                p = train_pairs[i]
                s1_list.append(p.s_pref)
                a1_list.append(p.a_pref)
                s2_list.append(p.s_rej)
                a2_list.append(p.a_rej)
                labels.append(1)
            if not labels:
                continue
            loss, _ = reward_model_loss(model, s1_list, a1_list, s2_list, a2_list, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if report_every > 0 and (epoch + 1) % report_every == 0:
            train_loss, train_acc = eval_reward_model(model, train_pairs)
            val_loss, val_acc = eval_reward_model(model, val_pairs)
            print(
                f"RM epoch {epoch + 1}/{epochs} | "
                f"train loss {train_loss:.3f} acc {train_acc:.3f} | "
                f"val loss {val_loss:.3f} acc {val_acc:.3f}"
            )
    return model

## Run experiments (across K + seeds)

In [ ]:
def infer_obs_act_dims(pairs: List[PrefPair]) -> Tuple[int, int, bool]:
    sample = pairs[0]
    obs_dim = sample.s_pref.shape[1] if sample.s_pref.ndim > 1 else sample.s_pref.shape[0]
    a = sample.a_pref
    if np.issubdtype(a.dtype, np.integer) and a.ndim == 1:
        action_dim = int(np.max(a)) + 1
        continuous = False
    else:
        continuous = True
        action_dim = a.shape[1] if a.ndim > 1 else 1
    return obs_dim, action_dim, continuous


def make_env(env_id: str, seed: int) -> gym.Env:
    env = gym.make(env_id)
    env.reset(seed=seed)
    env = Monitor(env)
    return env


class RewardModelEnv(gym.Wrapper):
    def __init__(self, env: gym.Env, reward_model: RewardModel, continuous: bool):
        super().__init__(env)
        self.reward_model = reward_model
        self.continuous = continuous
        self._last_obs = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._last_obs = obs
        return obs, info

    def step(self, action):
        obs_next, _, terminated, truncated, info = self.env.step(action)
        obs_t = torch.tensor(self._last_obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        if self.continuous:
            act_t = torch.tensor(np.asarray(action), dtype=torch.float32, device=DEVICE).unsqueeze(0)
        else:
            act_t = torch.tensor(np.asarray(action), dtype=torch.long, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            reward = float(self.reward_model(obs_t, act_t).cpu().item())
        self._last_obs = obs_next
        return obs_next, reward, terminated, truncated, info


def eval_policy_net(policy: PolicyNet, env_id: str, episodes: int, continuous: bool) -> float:
    env = make_env(env_id, seed=0)
    returns = []
    for _ in range(episodes):
        obs, _ = env.reset()
        done = False
        total = 0.0
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            if continuous:
                mean, _ = policy(obs_t)
                action = mean.squeeze(0).cpu().numpy()
            else:
                logits = policy(obs_t)
                action = int(torch.argmax(logits, dim=-1).item())
            obs, reward, terminated, truncated, _ = env.step(action)
            total += float(reward)
            done = terminated or truncated
        returns.append(total)
    env.close()
    return float(np.mean(returns))


def train_ppo_with_reward_model(
    env_id: str,
    reward_model: RewardModel,
    continuous: bool,
    seed: int,
) -> PPO:
    env = make_env(env_id, seed=seed)
    env = RewardModelEnv(env, reward_model, continuous)
    vec_env = DummyVecEnv([lambda: env])
    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=PPO_LR,
        n_steps=PPO_N_STEPS,
        batch_size=PPO_BATCH_SIZE,
        gamma=PPO_GAMMA,
        seed=seed,
        verbose=0,
    )
    model.learn(total_timesteps=PPO_TIMESTEPS)
    return model


def eval_policy_true_reward(model: PPO, env_id: str, episodes: int) -> float:
    env = DummyVecEnv([lambda: make_env(env_id, seed=0)])
    mean, _ = evaluate_policy(model, env, n_eval_episodes=episodes, deterministic=True)
    env.close()
    return float(mean)


results = []
if not dataset_list:
    print("No preference datasets found for", ENV_KEY)
else:
    for dataset in dataset_list:
        if not dataset.pairs:
            continue
        obs_dim, action_dim, continuous = infer_obs_act_dims(dataset.pairs)
        seed = dataset.seed
        torch.manual_seed(seed)
        np.random.seed(seed)
        dpo_policy = train_dpo(dataset.pairs, obs_dim, action_dim, continuous)
        dpo_score = eval_policy_net(dpo_policy, ENV_ID, EVAL_EPISODES, continuous)
        reward_model = train_reward_model(dataset.pairs, obs_dim, action_dim, continuous, seed=seed)
        ppo_model = train_ppo_with_reward_model(ENV_ID, reward_model, continuous, seed)
        ppo_score = eval_policy_true_reward(ppo_model, ENV_ID, EVAL_EPISODES)
        results.append({
            "env": dataset.env_key,
            "dataset": dataset.path.name,
            "K": dataset.k,
            "seed": seed,
            "dpo_return": dpo_score,
            "ppo_return": ppo_score,
        })
    if results:
        import pandas as pd
        df = pd.DataFrame(results)
        out_path = OUTPUT_DIR / f"results_{ENV_KEY}.csv"
        df.to_csv(out_path, index=False)
        print("saved:", out_path)

## Plot performance vs K

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


if "df" not in globals():
    result_path = OUTPUT_DIR / f"results_{ENV_KEY}.csv"
    if result_path.exists():
        df = pd.read_csv(result_path)


if "df" in globals() and not df.empty:
    plt.figure(figsize=(8, 4))
    for algo, col in [("DPO", "dpo_return"), ("PPO-RLHF", "ppo_return")]:
        sub = df.groupby("K")[col].mean().reset_index()
        plt.plot(sub["K"], sub[col], marker="o", label=algo)
    plt.xlabel("K (preference dataset size)")
    plt.ylabel("Average return")
    plt.title(f"{ENV_KEY} performance vs K")
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print("No results available to plot.")